# Percentage Calculation

Global area-weighted percentage differences in ET components between scenarios.

Data source: 5-year averaged (2010-2014) CLM5 output from `G:/Hangkai/CTH_ET project/Final_data/`

In [ ]:
import numpy as np
import netCDF4
import os
import pandas as pd

# ============================================================
# Percentage calculation using 5-year averaged data
# (averaged over 2010-2014)
# ============================================================
file_path = r'G:\Hangkai\CTH_ET project\Final_data'

scenario_mapping_EX = {
    'CLM Default': 'CLM Default.nc',
    'GEDI Max': 'GEDI Max.nc',
    'GEDI Mean': 'GEDI Mean.nc',
    'GEDI Median': 'GEDI Median.nc',
}

# Load PFT structure
with netCDF4.Dataset(os.path.join(file_path, 'CLM Default.nc'), 'r') as ds:
    pfts1d_wtgcell = np.array(ds.variables['pfts1d_wtgcell'][:])
    area = np.array(ds.variables['area'][:])

if pfts1d_wtgcell.ndim > 1:
    pfts1d_wtgcell = pfts1d_wtgcell[0, :]

# Load ET data for all scenarios
variables = ['FCEV', 'FCTR', 'FGEV']
data_EX = {}

for scenario in scenario_mapping_EX:
    data_EX[scenario] = {}
    nc_file = os.path.join(file_path, scenario_mapping_EX[scenario])
    with netCDF4.Dataset(nc_file, 'r') as ds:
        for variable in variables:
            data_EX[scenario][variable] = ds.variables[variable][:]

# Calculate area-weighted global means for each scenario
weighted_means = {}

for scenario in scenario_mapping_EX:
    weighted_means[scenario] = {}
    for var in variables:
        variable_values = np.mean(data_EX[scenario][var], axis=0) * pfts1d_wtgcell
        weighted_means[scenario][var] = np.sum(variable_values)

    # ET = FCEV + FCTR + FGEV
    weighted_means[scenario]['ET'] = (weighted_means[scenario]['FCEV'] +
                                      weighted_means[scenario]['FCTR'] +
                                      weighted_means[scenario]['FGEV'])

# Calculate percentage differences
combinations = [
    ('GEDI Mean', 'CLM Default'),
    ('GEDI Max', 'CLM Default'),
    ('GEDI Max', 'GEDI Mean'),
    ('GEDI Median', 'GEDI Mean')
]

results_percentage = {}

print("\n" + "="*70)
print("Global Area-Weighted Percentage Differences (5-year average)")
print("="*70)

for scenario1, scenario2 in combinations:
    results_percentage[(scenario1, scenario2)] = {}
    print(f"\n{scenario1} vs {scenario2}:")
    for var in ['FCEV', 'FCTR', 'FGEV', 'ET']:
        diff_percentage = 100 * (weighted_means[scenario1][var] - weighted_means[scenario2][var]) / weighted_means[scenario2][var]
        results_percentage[(scenario1, scenario2)][var] = diff_percentage
        print(f"  {var}: {diff_percentage:+.2f}%")

# Display as DataFrame
df = pd.DataFrame(results_percentage).T
df = df[['FCEV', 'FCTR', 'FGEV', 'ET']]
df.columns = ['FCEV (%)', 'FCTR (%)', 'FGEV (%)', 'ET (%)']
df.index.names = ['Scenario 1', 'Scenario 2']
print("\n")
print(df.to_string())